# 07 — Simulation and Counterfactuals

This notebook moves from prediction into scenario analysis.

The goal is to use Monte Carlo simulation to explore:
- season outcomes under different reliability assumptions
- podium and points probabilities under alternative qualifying outcomes
- constructor standings under weather or pit-stop shock scenarios

A key distinction in this notebook is the difference between:
- predictive simulation
- counterfactual scenario analysis
- causal interpretation

Not every hypothetical scenario is causal. Before simulating “what if” worlds, I first define which questions are descriptive, which are predictive, and which require stronger identification assumptions.

How much do starting conditions, reliability, and race shocks change expected season outcomes—and which of those changes can be interpreted causally? We can answer this by testing the following:

simulate season standings
simulate podium / points probabilities
test alternative qualifying outcomes
explore weather / pit shock scenarios
discuss causal limits honestly

## Defining the Causal Question

The most credible causal question in this notebook is:

**What is the effect of starting position on expected race outcomes, conditional on underlying driver and constructor strength? Holding driver and constructor strength constant?**

This question is useful because it connects naturally to:
- qualifying performance
- race simulation
- podium and points probabilities
- counterfactual starting positions

It also creates a more defensible foundation for Monte Carlo scenario analysis than jumping directly into broad race-day hypotheticals.

How do reliability shocks affect expected constructor standings over a season?

## Predictive vs Counterfactual vs Causal

This notebook uses three related but distinct ideas:

- **Predictive simulation:** drawing race or season outcomes from estimated probabilities
- **Counterfactual scenario analysis:** changing inputs such as grid position or reliability and re-simulating outcomes
- **Causal interpretation:** asking whether the simulated change can be interpreted as the effect of an intervention

These are not interchangeable.

For example:
- simulating different qualifying outcomes is a useful counterfactual exercise
- but calling that a causal estimate requires stronger assumptions about what generated those qualifying changes

## Section 1: Data Inputs for Simulation and Setup

- race-level expected finish or probability estimates
- points expectations
- driver / constructor identifiers
- grid / qualifying info
- reliability proxies
- weather flags
- pit-stop shock parameters

-----------------------------------------------------

- predicted finish position
- predicted points
- podium probability
- points probability
- DNF probability if available from prior work

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone

## Load Simulation Base Table

Use the cleaned pre-race feature store and model outputs from earlier notebooks as inputs into the simulation framework. What distribution are we simulating from?

In [ ]:
SIM_FEATURES = [
    "year",
    "round",
    "raceId",
    "driverId",
    "constructorId",
    "qualifying_position",
    "grid_clean",
    "driver_avg_finish_last5",
    "driver_dnf_rate_last5",
    "constructor_avg_finish_last5",
    "constructor_dnf_rate_last5",
    "is_wet_race",
    "alt",
    "temp_avg",
    "precipitation",
    "wind_speed"
]

## Section 2 Monte Carlo Simulation Design

If reliability improves or worsens for a team, how do season standings change?

For each race-driver:

- simulate finish / DNF using modeled expectations
- adjust DNF probability under scenarios
- convert to points
- aggregate over season
- repeat many times

If a driver starts 3 places higher, how do podium / points probabilities change?

- perturb grid_clean or qualifying_position
- recompute modeled probabilities
- simulate race outcomes many times
- compare to baseline

How sensitive are constructor standings to adverse weather or pit-stop shocks?

- introduce scenario-level shock distributions
- apply to subsets of races / constructors
- resimulate season points

## Monte Carlo Simulation: Season Outcomes Under Reliability Assumptions

This section simulates season standings under different reliability assumptions by altering DNF probabilities and re-running race outcomes thousands of times.

Scenarios:
- baseline
- +10% reliability improvement for constructor
- -10% reliability
- selective reliability shock to one driver

Outputs:
- championship probability
- average constructor points
- standing distributions

## Counterfactual Starting Positions

This section asks how podium and points probabilities change when starting position is shifted upward or downward while holding other pre-race features fixed.

Scenarios:

- driver starts +2 grid spots better
- driver starts +5 spots worse
- constructor average grid improves by 1 position all season

Outputs:

- podium probability difference
- expected points difference
- season standing changes

## Weather and Pit-Stop Shock Scenarios

This section introduces exogenous scenario shocks to examine how sensitive season outcomes are to added race-day volatility.

Scenarios:

- more wet races
- pit-stop penalty shock
- one catastrophic pit-stop event in key races
- repeated mild pit degradation

## Instrumental Variables: A Possible Future Direction

A fully causal interpretation of counterfactual starting positions would require stronger identification than is currently available in this dataset.

One possible future direction is to explore instrumental variables based on exogenous grid shocks, such as penalties or irregular qualifying disruptions. However, a credible instrument would need to affect starting position without directly affecting race outcomes through other channels.

At this stage, the notebook focuses on predictive counterfactuals rather than fully identified causal estimates.

Potentially:

- weather shock affecting qualifying or starting position but not race outcome except through starting position
** hard to defend because weather likely affects race outcomes directly too.

Another possible instrument:

- qualifying penalties / grid penalties
** may correlate with team reliability or strategy, so also not clean.

## Must-have visuals
1. Simulated constructor standings distribution
violin / box plots by scenario
2. Championship probability bars
baseline vs reliability scenarios
3. Points / podium probability change under grid shifts
line plot of probability vs starting position change
4. Season simulation fan chart
simulated points trajectories
5. Scenario comparison heatmap
constructors × scenarios
6. Counterfactual uplift plot
expected points gain from better starting positions

## How much of a season is shaped by what should have happened, and how much changes when we intervene on starting position, reliability, or race-day shocks?